In [ ]:
import torch
print(torch.__version__)

In [ ]:
# GPU-enabled install (Colab)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# CPU-only install (if no GPU)
# !pip install torch torchvision torchaudio

# Install/upgrade Transformers, Pandas, tqdm
!pip install -U transformers pandas tqdm


In [ ]:
import torch
import sys

print("Python version:", sys.version)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

MODEL = "Ishan0612/biobert-ner-disease-ncbi"  # or "d4data/biomedical-ner-all"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForTokenClassification.from_pretrained(MODEL)

ner_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)


In [ ]:
text = "The patient was treated with aspirin for myocardial infarction."
entities = ner_pipeline(text)

for e in entities:
    print(f"{e['word']} → {e['entity_group']} (score: {e['score']:.2f})")


In [ ]:
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

# ----------------------------
# Paths and settings
# ----------------------------
input_csv = "./data/BioBERT_NER/preprocessed_data_ner.csv"
text_column = "Preprocessed Posts"
output_csv = input_csv.replace(".csv", "_with_entities.csv")

MODEL = "Ishan0612/biobert-ner-disease-ncbi"

# ----------------------------
# Load model and tokenizer
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForTokenClassification.from_pretrained(MODEL).to(device)
model.eval()

id2label = model.config.id2label
max_len = tokenizer.model_max_length  # usually 512

# ----------------------------
# Helper to decode tokens into entities
# ----------------------------
def decode_entities(tokens, preds):
    entities = []
    current_word = ""
    current_label = None
    for token, pred_id in zip(tokens, preds):
        label = id2label[pred_id]
        if label == "O":
            if current_word:
                entities.append(f"{current_word} ({current_label})")
                current_word = ""
                current_label = None
            continue
        if token.startswith("##"):
            current_word += token[2:]
        else:
            if current_word:
                entities.append(f"{current_word} ({current_label})")
            current_word = token
            current_label = label
    if current_word:
        entities.append(f"{current_word} ({current_label})")
    return entities


# ----------------------------
# Robust text processor (auto chunking)
# ----------------------------
def process_text(text):
    tokens = tokenizer.tokenize(text)
    entities = []
    for i in range(0, len(tokens), max_len - 2):
        chunk_tokens = tokens[i:i + max_len - 2]
        chunk_text = tokenizer.convert_tokens_to_string(chunk_tokens)

        # Encode safely
        encoding = tokenizer.encode_plus(
            chunk_text,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoding)
            predictions = torch.argmax(outputs.logits, dim=-1)[0].tolist()

        tokens_decoded = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])
        entities += decode_entities(tokens_decoded, predictions)

    return "; ".join(entities)


# ----------------------------
# Process CSV
# ----------------------------
df = pd.read_csv(input_csv)
all_entities = []

for text in tqdm(df[text_column].astype(str), total=len(df), desc="Extracting entities"):
    try:
        ents = process_text(text)
    except Exception as e:
        print(f"⚠️ Skipping text due to error: {e}")
        ents = ""
    all_entities.append(ents)

df["recognized_entities"] = all_entities
df.to_csv(output_csv, index=False)
print(f"✅ Finished. Saved to: {output_csv}")


In [ ]:
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

# ----------------------------
# Paths and settings
# ----------------------------
input_csv = "./data/BioBERT_NER/preprocessed_data_ner.csv"
text_column = "Preprocessed Posts"
output_csv = input_csv.replace(".csv", "_with_entities.csv")

MODEL = "Ishan0612/biobert-ner-disease-ncbi"

# ----------------------------
# Load model and tokenizer
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForTokenClassification.from_pretrained(MODEL).to(device)
model.eval()

id2label = model.config.id2label
max_len = tokenizer.model_max_length  # usually 512

# ----------------------------
# Decode token predictions into readable entities
# ----------------------------
def decode_entities(tokens, preds):
    entities = []
    current_word = ""
    current_label = None
    for token, pred_id in zip(tokens, preds):
        label = id2label[pred_id]
        if label == "O":
            if current_word:
                entities.append(f"{current_word} ({current_label})")
                current_word = ""
                current_label = None
            continue
        if token.startswith("##"):
            current_word += token[2:]
        else:
            if current_word:
                entities.append(f"{current_word} ({current_label})")
            current_word = token
            current_label = label
    if current_word:
        entities.append(f"{current_word} ({current_label})")
    return entities

# ----------------------------
# Function to process very long text
# ----------------------------
def process_long_text(text):
    tokens = tokenizer.tokenize(text)
    all_entities = []

    # process in chunks of <=510 tokens
    for i in range(0, len(tokens), max_len - 2):
        chunk_tokens = tokens[i:i + max_len - 2]
        chunk_text = tokenizer.convert_tokens_to_string(chunk_tokens)

        # encode this chunk safely
        encoding = tokenizer(
            chunk_text,
            return_tensors="pt",
            truncation=True,
            max_length=max_len,
            add_special_tokens=True,
            return_attention_mask=True
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoding)
            preds = torch.argmax(outputs.logits, dim=-1)[0].tolist()

        chunk_tokens_decoded = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])
        chunk_entities = decode_entities(chunk_tokens_decoded, preds)
        all_entities.extend(chunk_entities)

    return "; ".join(all_entities)

# ----------------------------
# Process CSV (no skipping)
# ----------------------------
df = pd.read_csv(input_csv)
recognized = []

for text in tqdm(df[text_column].astype(str), total=len(df), desc="Extracting entities"):
    ents = process_long_text(text)
    recognized.append(ents)

df["recognized_entities"] = recognized
df.to_csv(output_csv, index=False)
print(f"✅ Finished. Saved to: {output_csv}")
